In [ ]:
# membandit setup
import os
import sys
from pathlib import Path

# Ensure cleaned_repo (parent of membandit/) is on sys.path when running from notebooks/
_here = Path.cwd().resolve()
for _candidate in [_here, *_here.parents]:
    if (_candidate / "membandit" / "__init__.py").exists():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

# Optional: set MEMORYBANDIT_DATA_DIR to your data root (defaults to /Volumes/T7 Shield)
# os.environ.setdefault("MEMORYBANDIT_DATA_DIR", "/path/to/data/root")

from membandit.plotting_style import set_plot_style
set_plot_style()


# ACC Cue-Period Theta Multivariate Imaginary Coherency Across Target Regions

Compute trial-wise multivariate maximised imaginary coherency (MIC) between `ACC` and each target ROI during the full **cue epoch (0–1 s)** and **theta (4–8 Hz)**.

Target regions: `HPC`, `AMY`, `dlPFC`, `dmPFC`, `OFC`.

The exported table contains shuffle-z-scored MIC (`mic_z`) plus trial/behavior metadata (`hit_miss`, `rpe_cat`, `region_pair`) for region-pair plots and mixed-effects modeling.


In [ ]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

import os
import json
import hashlib
import warnings
from copy import deepcopy
from itertools import combinations
from typing import Any, Dict, List

import mne
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
from joblib import Parallel, delayed
from scipy.stats import chi2, ttest_rel
from statsmodels.stats.multitest import multipletests
from IPython.display import display

from membandit.plotting_style import set_plot_style
from membandit.behavior import load_behavior_tables, preprocess_learn_df, preprocess_combined_df
from membandit.io import load_subject_epochs
from membandit.granger import build_hit_miss_trial_map

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, **kwargs):
        return x

try:
    from mne_connectivity import spectral_connectivity_time
except Exception:
    spectral_connectivity_time = None

warnings.filterwarnings('ignore')
set_plot_style()
sns.set_style('whitegrid')
np.set_printoptions(suppress=True, precision=4)


In [ ]:
# Paths and subject list
base_dir = '/Volumes/T7 Shield'
load_dir = f'{base_dir}/work/qasims01/MemoryBanditData/EMU'
savepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/Connectivity/scratch/mvmic'
os.makedirs(savepath, exist_ok=True)

demo_file = f'{base_dir}/projects/guLab/Salman/EMU/MemoryBanditsDatabase.xlsx'
demo_data = pd.read_excel(demo_file, engine='openpyxl')
subj_df = demo_data[(demo_data.LongVersion == 1) & (demo_data.Minerva == 1)].copy()
subj_df = subj_df[~subj_df['MSSMCode'].isin(['MS015', 'MS024', 'MS034', 'MS038'])]
subjects = sorted(subj_df['MSSMCode'].dropna().unique().tolist())

learn_df_raw, combined_df_raw = load_behavior_tables(load_dir)
learn_df = preprocess_learn_df(learn_df_raw)
combined_df = preprocess_combined_df(combined_df_raw)

print('Subjects:', len(subjects))
print('combined_df:', combined_df.shape)
print('combined_df has RPE metadata:', {'participant', 'rpe', 'rpe_cat'}.issubset(combined_df.columns))


In [ ]:
# Analysis configuration
MIC_CFG = {
    'roi_source': 'ACC',
    'target_rois': ['HPC', 'AMY', 'dlPFC', 'dmPFC', 'OFC'],
    'events': {
        'cue_on': {'tmin': 0.0, 'tmax': 1.0},
    },
    'day': 'Day1',
    'target_sfreq': 50,
    'fmin': 4,
    'fmax': 8,
    'n_freqs': 8,
    'n_cycles': 3,
    'random_state': 42,
    'min_shift': 2,
}

ACC_ROI = MIC_CFG['roi_source']
REGION_PAIR_ORDER = [f"{ACC_ROI}-{roi}" for roi in MIC_CFG['target_rois']]
CUE_WINDOW = (0.0, 1.0)
MIN_TRIALS_PER_MODEL = 10

RUN = {
    'run_single_subject_smoke': True,
    'run_batch': True,
    'overwrite_subject_cache': False,
    'smoke_subject': 'UI002' if 'UI002' in subjects else subjects[0],
    'smoke_target_rois': MIC_CFG['target_rois'],
    'n_surrogates_smoke': 25,
    'n_surrogates_batch': 100,
    'n_jobs_batch': 6,
    'parallel_backend': 'loky',
}

ANALYSIS_STATE = globals().get('ANALYSIS_STATE', {})

def _analysis_value_token(value):
    return str(value).replace('.', 'p').replace('-', 'm')

def _stable_json_dumps(payload):
    return json.dumps(payload, sort_keys=True, separators=(',', ':'), default=str)

def _build_analysis_tag(cfg):
    cue = cfg['events'].get('cue_on', {})
    key = {
        'roi_source': cfg['roi_source'],
        'target_rois': cfg['target_rois'],
        'events': cfg['events'],
        'day': cfg['day'],
        'target_sfreq': cfg['target_sfreq'],
        'fmin': cfg['fmin'],
        'fmax': cfg['fmax'],
        'n_freqs': cfg['n_freqs'],
        'n_cycles': cfg['n_cycles'],
    }
    cfg_hash = hashlib.sha1(_stable_json_dumps(key).encode('utf-8')).hexdigest()[:10]
    cue_tmin = _analysis_value_token(cue.get('tmin', 'na'))
    cue_tmax = _analysis_value_token(cue.get('tmax', 'na'))
    return (
        f"trialwise_{cfg['roi_source']}toTargets_{cfg['day']}"
        f"_sf{_analysis_value_token(cfg['target_sfreq'])}"
        f"_{cfg['fmin']}-{cfg['fmax']}Hz"
        f"_cue{cue_tmin}-{cue_tmax}"
        f"_roi{cfg_hash}"
    )

analysis_tag = _build_analysis_tag(MIC_CFG)
analysis_dir = os.path.join(savepath, 'notebook_runs', analysis_tag)
subject_cache_dir = os.path.join(analysis_dir, 'subject_cache')
os.makedirs(subject_cache_dir, exist_ok=True)

analysis_paths = {
    'analysis_dir': analysis_dir,
    'subject_cache_dir': subject_cache_dir,
    'combined_csv': os.path.join(analysis_dir, 'trialwise_acc_cue_mic_z.csv'),
    'cache_log_csv': os.path.join(analysis_dir, 'cache_log.csv'),
    'summary_csv': os.path.join(analysis_dir, 'trialwise_acc_cue_subject_summary.csv'),
    'region_pair_pdf': os.path.join(analysis_dir, 'window_mic_by_region_pair.pdf'),
    'region_pair_png': os.path.join(analysis_dir, 'window_mic_by_region_pair.png'),
    'hit_rpe_pdf': os.path.join(analysis_dir, 'window_mic_by_region_hit_rpe.pdf'),
    'hit_rpe_png': os.path.join(analysis_dir, 'window_mic_by_region_hit_rpe.png'),
    'mixedlm_summary_txt': os.path.join(analysis_dir, 'mixedlm_summary.txt'),
    'mixedlm_fixed_effects_csv': os.path.join(analysis_dir, 'mixedlm_fixed_effects.csv'),
    'mixedlm_omnibus_csv': os.path.join(analysis_dir, 'mixedlm_omnibus_tests.csv'),
    'pairwise_region_csv': os.path.join(analysis_dir, 'pairwise_region_contrasts.csv'),
    'condition_summary_csv': os.path.join(analysis_dir, 'condition_stratified_summary.csv'),
}

print('Smoke subject:', RUN['smoke_subject'])
print('Smoke targets:', RUN['smoke_target_rois'])
print('Cue window:', CUE_WINDOW)
print('Region pairs:', REGION_PAIR_ORDER)
print('Analysis dir:', analysis_dir)
print('Batch enabled:', RUN['run_batch'])



## Helper Functions

Null distribution: circularly shift only the source/ACC channels inside each trial, then recompute trial-wise MIC.


In [ ]:
def resolve_trial_col_for_event(df, event):
    event_name = str(event).strip().lower()
    if event_name == 'cue_on':
        candidates = ('trials_mem', 'trials_gamble', 'trials')
    else:
        candidates = ('trials_gamble', 'trials', 'trials_mem')
    for col in candidates:
        if col in df.columns:
            return col
    raise ValueError(f'Could not resolve trial column for event={event!r}')

def normalize_rpe_cat(s):
    mapping = {
        'negative': 'neg',
        'neg': 'neg',
        'neutral': 'neutral',
        'positive': 'pos',
        'pos': 'pos',
    }
    return s.astype(str).str.strip().str.lower().map(mapping)

def build_trial_map(combined_df, subject, event, n_epochs, day='Day1'):
    trial_col = resolve_trial_col_for_event(combined_df, event)
    trial_map = combined_df[combined_df['participant'].eq(subject)].copy()
    if 'condition' in trial_map.columns:
        trial_map = trial_map[trial_map['condition'].eq(day)].copy()
    trial_map = trial_map.dropna(subset=[trial_col]).copy()
    if len(trial_map) == 0:
        return pd.DataFrame(), trial_col

    trial_map['trial_index'] = pd.to_numeric(trial_map[trial_col], errors='coerce').astype('Int64') - 1
    trial_map = trial_map.dropna(subset=['trial_index']).copy()
    trial_map['trial_index'] = trial_map['trial_index'].astype(int)
    trial_map = trial_map[(trial_map['trial_index'] >= 0) & (trial_map['trial_index'] < int(n_epochs))].copy()
    trial_map = trial_map.sort_values('trial_index').drop_duplicates(subset=['trial_index'], keep='first').reset_index(drop=True)

    if 'rpe_cat' in trial_map.columns:
        trial_map['rpe_cat'] = normalize_rpe_cat(trial_map['rpe_cat'])

    for col in ['trials_mem', 'trials_gamble', 'trials', 'rpe', 'rpe_cat', 'memory_outcome', 'hit_miss']:
        if col not in trial_map.columns:
            trial_map[col] = np.nan

    keep = ['participant', 'condition', 'trial_index', 'trials_mem', 'trials_gamble', 'trials', 'rpe', 'rpe_cat', 'memory_outcome', 'hit_miss']
    return trial_map[[c for c in keep if c in trial_map.columns]].reset_index(drop=True), trial_col



def merge_hit_miss_into_trial_map(trial_map, subject, event, n_epochs, day='Day1'):
    hm_map = build_hit_miss_trial_map(
        combined_df,
        subj_id=subject,
        n_epochs=int(n_epochs),
        day=day,
        event=event,
    )
    if len(hm_map) == 0:
        trial_map['hit_miss'] = np.nan
        return trial_map
    hm_map = hm_map[hm_map['alignment_valid']].copy()
    hm = hm_map[['epoch_idx', 'hit_miss']].rename(columns={'epoch_idx': 'trial_index'})
    out = trial_map.merge(hm, on='trial_index', how='left')
    if 'hit_miss' not in out.columns:
        out['hit_miss'] = np.nan
    return out

def make_multivariate_indices(source_idx, target_idx):
    source_idx = np.unique(np.asarray(source_idx, dtype=int))
    target_idx = np.unique(np.asarray(target_idx, dtype=int))
    return (np.array([source_idx.tolist()], dtype=object), np.array([target_idx.tolist()], dtype=object))

def extract_trial_mic_vector(con_obj, n_epochs, label):
    raw = np.asarray(con_obj.get_data())
    arr = np.squeeze(raw)
    arr = np.abs(arr.astype(complex, copy=False))

    if arr.ndim == 0:
        raise RuntimeError(f'{label}: expected one value per trial, got scalar')

    matching_axes = [axis for axis, size in enumerate(arr.shape) if size == int(n_epochs)]
    if len(matching_axes) == 0:
        raise RuntimeError(f'{label}: expected an axis with n_epochs={n_epochs}, got shape {arr.shape}')

    # Prefer the first matching axis, then average any remaining connection/frequency/time dimensions.
    arr = np.moveaxis(arr, matching_axes[0], 0)
    if arr.shape[0] != int(n_epochs):
        raise RuntimeError(f'{label}: trial axis move failed, got shape {arr.shape}')
    return np.nanmean(arr.reshape(int(n_epochs), -1), axis=1).astype(float)

def scalar_null_stats(real_value, surrogate_values, eps=1e-8):
    surrogate_values = np.asarray(surrogate_values, dtype=float)
    surrogate_values = surrogate_values[np.isfinite(surrogate_values)]
    if surrogate_values.size == 0 or not np.isfinite(real_value):
        return dict(z=np.nan, n=0)
    mean_val = float(np.nanmean(surrogate_values))
    std_val = float(max(np.nanstd(surrogate_values), eps))
    return dict(z=float((real_value - mean_val) / std_val), n=int(surrogate_values.size))

def stable_int_seed(*parts, modulo=1_000_000):
    token = _stable_json_dumps({'parts': [str(p) for p in parts]})
    return int(hashlib.sha1(token.encode('utf-8')).hexdigest()[:12], 16) % int(modulo)

def effective_n_cycles(epochs, cfg, safety=0.55):
    # MNE's Morlet wavelet spans roughly 10 sigma, so low-frequency wavelets
    # can exceed short cue epochs. Keep the requested value when safe, otherwise
    # reduce cycles enough for the shortest frequency to fit inside the epoch.
    duration = float(epochs.times[-1] - epochs.times[0]) if len(epochs.times) > 1 else 0.0
    if duration <= 0:
        return max(1.0, min(float(cfg['n_cycles']), 2.0))
    max_cycles = safety * duration * float(cfg['fmin'])
    return float(max(1.0, min(float(cfg['n_cycles']), max_cycles)))

def compute_trialwise_mic(epochs, indices, freqs, cfg, label='mic'):
    n_cycles_eff = effective_n_cycles(epochs, cfg)
    con = spectral_connectivity_time(
        epochs,
        sfreq=float(epochs.info['sfreq']),
        method=['mic'],
        indices=indices,
        fmin=cfg['fmin'],
        fmax=cfg['fmax'],
        freqs=freqs,
        n_cycles=n_cycles_eff,
        average=False,
        faverage=True,
        verbose='ERROR',
    )
    return extract_trial_mic_vector(con, len(epochs), label)

def shifted_source_epochs(epochs, source_idx, rng, min_shift=2):
    base_data = epochs.get_data(copy=True)
    source_idx = np.unique(np.asarray(source_idx, dtype=int))
    n_trials, _, n_times = base_data.shape
    candidate_shifts = np.arange(int(min_shift), n_times - int(min_shift))
    if candidate_shifts.size == 0:
        candidate_shifts = np.arange(1, n_times)
    if candidate_shifts.size == 0:
        raise RuntimeError('Cannot circular-shift source data with fewer than two timepoints')

    for trial_idx in range(n_trials):
        shift = int(rng.choice(candidate_shifts))
        base_data[trial_idx, source_idx, :] = np.roll(base_data[trial_idx, source_idx, :], shift=shift, axis=-1)

    return mne.EpochsArray(
        base_data,
        info=deepcopy(epochs.info),
        events=epochs.events.copy(),
        event_id=epochs.event_id.copy() if epochs.event_id is not None else None,
        tmin=float(epochs.tmin),
        baseline=epochs.baseline,
        verbose=False,
    )



In [ ]:
def compute_subject_event_target_mic(subject, event, target_roi, n_surrogates, verbose=False):
    epoch_window = MIC_CFG['events'][event]
    epoch_data = load_subject_epochs(
        subj_id=subject,
        base_dir=base_dir,
        event=event,
        roi_source=MIC_CFG['roi_source'],
        roi_target=target_roi,
        tmin=epoch_window['tmin'],
        tmax=epoch_window['tmax'],
        target_sfreq=MIC_CFG['target_sfreq'],
        day=MIC_CFG['day'],
        fmin=MIC_CFG['fmin'],
        fmax=MIC_CFG['fmax'],
    )
    if epoch_data is None:
        return pd.DataFrame(), {'subject': subject, 'event': event, 'roi_target': target_roi, 'skip_reason': 'epoch_load_failed'}

    epochs = epoch_data['epochs']
    trial_map, trial_col = build_trial_map(combined_df, subject, event, len(epochs), day=MIC_CFG['day'])
    trial_map = merge_hit_miss_into_trial_map(trial_map, subject, event, len(epochs), day=MIC_CFG['day'])
    if len(trial_map) == 0:
        return pd.DataFrame(), {'subject': subject, 'event': event, 'roi_target': target_roi, 'skip_reason': 'no_aligned_trials'}

    freqs = np.linspace(MIC_CFG['fmin'], MIC_CFG['fmax'], int(MIC_CFG['n_freqs']))
    rng = np.random.default_rng(
        MIC_CFG['random_state'] + stable_int_seed(subject, event, target_roi)
    )

    real_by_hemi = {}
    surr_by_hemi = {}
    for hemi in ('l', 'r'):
        source_idx = np.unique(np.asarray(epoch_data.get(f'source_indices_{hemi}', []), dtype=int))
        target_idx = np.unique(np.asarray(epoch_data.get(f'target_indices_{hemi}', []), dtype=int))
        if source_idx.size == 0 or target_idx.size == 0:
            continue

        indices = make_multivariate_indices(source_idx, target_idx)
        try:
            real = compute_trialwise_mic(epochs, indices, freqs, MIC_CFG, label=f'{subject} {event} {target_roi} {hemi} real')
        except Exception as exc:
            if verbose:
                print(f'{subject} {event} {target_roi} {hemi}: real MIC failed:', exc)
            continue

        surr = np.full((int(n_surrogates), len(epochs)), np.nan, dtype=float)
        iterator = range(int(n_surrogates))
        if verbose:
            iterator = tqdm(iterator, desc=f'{subject} {event} {target_roi} {hemi} surrogates')
        for s_idx in iterator:
            try:
                shifted_epochs = shifted_source_epochs(
                    epochs,
                    source_idx=source_idx,
                    rng=rng,
                    min_shift=MIC_CFG['min_shift'],
                )
                surr[s_idx] = compute_trialwise_mic(
                    shifted_epochs,
                    indices,
                    freqs,
                    MIC_CFG,
                    label=f'{subject} {event} {target_roi} {hemi} surr {s_idx}',
                )
            except Exception as exc:
                if verbose:
                    print(f'{subject} {event} {target_roi} {hemi}: surrogate {s_idx} failed:', exc)
                continue

        valid_surrogates = int(np.isfinite(surr).any(axis=1).sum())
        if valid_surrogates == 0:
            if verbose:
                print(f'{subject} {event} {target_roi} {hemi}: all surrogates failed')
            continue

        real_by_hemi[hemi] = real
        surr_by_hemi[hemi] = surr

    if len(real_by_hemi) == 0:
        return pd.DataFrame(), {'subject': subject, 'event': event, 'roi_target': target_roi, 'skip_reason': 'no_valid_hemispheres'}

    rows = []
    for _, trial in trial_map.iterrows():
        trial_idx = int(trial['trial_index'])
        hemi_rows = []
        for hemi in real_by_hemi:
            stats = scalar_null_stats(real_by_hemi[hemi][trial_idx], surr_by_hemi[hemi][:, trial_idx])
            if np.isfinite(stats['z']):
                hemi_rows.append(stats)
        if len(hemi_rows) == 0:
            continue

        rows.append({
            'participant': subject,
            'event': event,
            'trial_index': trial_idx,
            'trials_mem': trial.get('trials_mem', np.nan),
            'trials_gamble': trial.get('trials_gamble', np.nan),
            'rpe': trial.get('rpe', np.nan),
            'rpe_cat': trial.get('rpe_cat', np.nan),
            'memory_outcome': trial.get('memory_outcome', np.nan),
            'hit_miss': trial.get('hit_miss', np.nan),
            'roi_source': MIC_CFG['roi_source'],
            'roi_target': target_roi,
            'region_pair': f"{MIC_CFG['roi_source']}-{target_roi}",
            'mic_z': float(np.nanmean([h['z'] for h in hemi_rows])),
            'n_surrogates': int(np.round(np.nanmean([h['n'] for h in hemi_rows]))),
            'valid_hemi_count': int(len(hemi_rows)),
            'trial_col': trial_col,
        })

    out_df = pd.DataFrame(rows).sort_values('trial_index').reset_index(drop=True)
    meta = {
        'subject': subject,
        'event': event,
        'roi_target': target_roi,
        'skip_reason': '' if len(out_df) > 0 else 'no_successful_trials',
        'rows': int(len(out_df)),
        'valid_hemispheres': sorted(real_by_hemi.keys()),
        'n_epochs': int(len(epochs)),
        'n_aligned_trials': int(len(trial_map)),
        'n_surrogates_requested': int(n_surrogates),
        'n_cycles_effective': float(effective_n_cycles(epochs, MIC_CFG)),
    }
    return out_df, meta

def subject_cache_path(subject, event, target_roi, n_surrogates):
    key = {
        'subject': subject,
        'event': event,
        'target_roi': target_roi,
        'cfg': MIC_CFG,
        'n_surrogates': int(n_surrogates),
    }
    h = hashlib.sha1(_stable_json_dumps(key).encode('utf-8')).hexdigest()[:12]
    fname = f'{subject}_{event}_{MIC_CFG["roi_source"]}to{target_roi}_{h}.csv.gz'
    return os.path.join(subject_cache_dir, fname)

def run_or_load_subject_event_target(subject, event, target_roi, n_surrogates, overwrite=False, verbose=False):
    cache_file = subject_cache_path(subject, event, target_roi, n_surrogates)
    if (not overwrite) and os.path.exists(cache_file):
        df = pd.read_csv(cache_file)
        meta = {'subject': subject, 'event': event, 'roi_target': target_roi, 'cache_hit': True, 'cache_file': cache_file, 'rows': int(len(df)), 'skip_reason': ''}
        return df, meta

    df, meta = compute_subject_event_target_mic(subject, event, target_roi, n_surrogates, verbose=verbose)
    meta['cache_hit'] = False
    meta['cache_file'] = cache_file
    if len(df) > 0:
        os.makedirs(os.path.dirname(cache_file), exist_ok=True)
        df.to_csv(cache_file, index=False, compression='gzip')
    return df, meta



## Smoke Test

Runs one subject across all target ROIs for `cue_on`. Finish this before enabling the full batch.


In [ ]:
assert spectral_connectivity_time is not None, 'mne_connectivity.spectral_connectivity_time is required for this notebook'

smoke_df = pd.DataFrame()
smoke_log = []
if RUN['run_single_subject_smoke']:
    smoke_frames = []
    for target_roi in RUN['smoke_target_rois']:
        df_ev, meta_ev = run_or_load_subject_event_target(
            subject=RUN['smoke_subject'],
            event='cue_on',
            target_roi=target_roi,
            n_surrogates=RUN['n_surrogates_smoke'],
            overwrite=RUN['overwrite_subject_cache'],
            verbose=True,
        )
        smoke_log.append(meta_ev)
        if len(df_ev) > 0:
            smoke_frames.append(df_ev)
    smoke_df = pd.concat(smoke_frames, ignore_index=True) if len(smoke_frames) > 0 else pd.DataFrame()

print('Smoke rows:', len(smoke_df))
display(pd.DataFrame(smoke_log))
if len(smoke_df) > 0:
    required_cols = {
        'participant', 'event', 'trial_index', 'rpe_cat', 'hit_miss', 'region_pair',
        'roi_source', 'roi_target', 'mic_z', 'n_surrogates', 'valid_hemi_count',
    }
    assert required_cols.issubset(smoke_df.columns), f'Missing columns: {required_cols - set(smoke_df.columns)}'
    assert smoke_df['mic_z'].notna().all(), 'Smoke mic_z must be finite for successful rows'
    assert (smoke_df['n_surrogates'] > 0).all(), 'Smoke n_surrogates must be positive'
    assert (smoke_df['valid_hemi_count'] >= 1).all(), 'Smoke valid_hemi_count must be >= 1'
    assert smoke_df['region_pair'].isin(REGION_PAIR_ORDER).all(), 'Unexpected region_pair labels'
    display(smoke_df.head())



## Batch Run

Set `RUN['run_batch'] = True` above to compute all subjects and all target ROIs for `cue_on`.


In [ ]:
all_mic_df = pd.DataFrame()
cache_log_df = pd.DataFrame()

if RUN['run_batch']:
    frames = []
    log_rows = []
    total_jobs = len(subjects) * len(MIC_CFG['events']) * len(MIC_CFG['target_rois'])
    iterator = tqdm(
        [(s, ev, roi) for s in subjects for ev in MIC_CFG['events'] for roi in MIC_CFG['target_rois']],
        total=total_jobs,
        desc='Trial-wise MIC batch',
    )
    for subject, event, target_roi in iterator:
        df_job, meta_job = run_or_load_subject_event_target(
            subject=subject,
            event=event,
            target_roi=target_roi,
            n_surrogates=RUN['n_surrogates_batch'],
            overwrite=RUN['overwrite_subject_cache'],
            verbose=False,
        )
        log_rows.append(meta_job)
        if len(df_job) > 0:
            frames.append(df_job)

    all_mic_df = pd.concat(frames, ignore_index=True) if len(frames) > 0 else pd.DataFrame()
    cache_log_df = pd.DataFrame(log_rows)
    cache_log_df.to_csv(analysis_paths['cache_log_csv'], index=False)
    if len(all_mic_df) > 0:
        all_mic_df.to_csv(analysis_paths['combined_csv'], index=False)
elif os.path.exists(analysis_paths['combined_csv']):
    all_mic_df = pd.read_csv(analysis_paths['combined_csv'])
    cache_log_df = pd.read_csv(analysis_paths['cache_log_csv']) if os.path.exists(analysis_paths['cache_log_csv']) else pd.DataFrame()
    print('Loaded existing combined cache:', analysis_paths['combined_csv'])
else:
    all_mic_df = smoke_df.copy()
    cache_log_df = pd.DataFrame(smoke_log)
    print('Batch disabled and no combined cache found; using smoke_df for downstream checks/plots.')

print('all_mic_df rows:', len(all_mic_df))
if len(cache_log_df) > 0:
    display(cache_log_df.head())
if len(all_mic_df) > 0:
    display(all_mic_df.head())



In [ ]:
if len(all_mic_df) == 0:
    print('No MIC rows available for checks.')
else:
    required_cols = [
        'participant', 'event', 'trial_index', 'trials_mem', 'trials_gamble', 'rpe', 'rpe_cat',
        'memory_outcome', 'hit_miss', 'region_pair', 'roi_source', 'roi_target', 'mic_z',
        'n_surrogates', 'valid_hemi_count',
    ]
    assert required_cols.issubset(all_mic_df.columns), f'Missing required columns: {required_cols - set(all_mic_df.columns)}'
    assert not all_mic_df.duplicated(['participant', 'event', 'roi_target', 'trial_index']).any(), 'Rows must be unique per participant/event/target/trial'
    assert all_mic_df['roi_source'].eq(MIC_CFG['roi_source']).all(), 'Unexpected source ROI'
    assert set(all_mic_df['event'].dropna().unique()).issubset(set(MIC_CFG['events'].keys())), 'Unexpected event in output'
    assert all_mic_df['mic_z'].notna().all(), 'Successful rows must have finite mic_z'
    assert (all_mic_df['n_surrogates'] > 0).all(), 'n_surrogates must be positive'
    assert (all_mic_df['valid_hemi_count'] >= 1).all(), 'valid_hemi_count must be >= 1'
    if RUN['run_batch'] or os.path.exists(analysis_paths['combined_csv']):
        present_events = set(all_mic_df['event'].dropna().unique())
        assert set(MIC_CFG['events'].keys()).issubset(present_events), f'Missing events: {set(MIC_CFG["events"].keys()) - present_events}'
    print('Integrity checks passed.')
    print('Rows:', len(all_mic_df))
    print('Subjects:', all_mic_df['participant'].nunique())
    print('Region pairs:', sorted(all_mic_df['region_pair'].dropna().unique().tolist()))



## Region-Pair Plots and Mixed-Effects Modeling

Requires `all_mic_df` from smoke test or batch run.


In [ ]:
def _term_mask(names: List[str], include: List[str], exact_colons: int) -> np.ndarray:
    out = []
    for name in names:
        if name == "Intercept":
            out.append(False)
            continue
        if name.count(":") != exact_colons:
            out.append(False)
            continue
        out.append(all(token in name for token in include))
    return np.asarray(out, dtype=bool)


def wald_test_from_mask(result, mask: np.ndarray, label: str) -> Dict[str, Any]:
    names = list(result.fe_params.index)
    if mask.sum() == 0:
        return {"term": label, "df": 0, "chi2": np.nan, "p_value": np.nan}

    beta = np.asarray(result.fe_params, dtype=float)
    cov = np.asarray(result.cov_params().loc[names, names], dtype=float)
    idx = np.flatnonzero(mask)
    b = beta[idx]
    V = cov[np.ix_(idx, idx)]
    try:
        stat = float(b.T @ np.linalg.pinv(V) @ b)
        p_value = float(chi2.sf(stat, df=len(idx)))
    except Exception:
        stat = np.nan
        p_value = np.nan
    return {"term": label, "df": int(len(idx)), "chi2": stat, "p_value": p_value}


def fit_primary_mixedlm(data: pd.DataFrame):
    formula = "mic_z ~ C(region_pair) * C(hit_miss) * C(rpe_cat)"
    methods = ["lbfgs", "powell", "cg"]
    last_error = None
    for method in methods:
        try:
            model = smf.mixedlm(formula=formula, data=data, groups=data["participant"])
            result = model.fit(reml=False, method=method, maxiter=500, disp=False)
            return result, method
        except Exception as exc:
            last_error = exc
    raise RuntimeError(f"MixedLM failed for all methods. Last error: {last_error}")


def build_omnibus_table(result) -> pd.DataFrame:
    names = list(result.fe_params.index)
    tests = [
        ("region_pair", _term_mask(names, ["C(region_pair)"], exact_colons=0)),
        ("hit_miss", _term_mask(names, ["C(hit_miss)"], exact_colons=0)),
        ("rpe_cat", _term_mask(names, ["C(rpe_cat)"], exact_colons=0)),
        ("region_pair:hit_miss", _term_mask(names, ["C(region_pair)", "C(hit_miss)"], exact_colons=1)),
        ("region_pair:rpe_cat", _term_mask(names, ["C(region_pair)", "C(rpe_cat)"], exact_colons=1)),
        ("hit_miss:rpe_cat", _term_mask(names, ["C(hit_miss)", "C(rpe_cat)"], exact_colons=1)),
        (
            "region_pair:hit_miss:rpe_cat",
            _term_mask(names, ["C(region_pair)", "C(hit_miss)", "C(rpe_cat)"], exact_colons=2),
        ),
    ]
    rows = [wald_test_from_mask(result, mask, label) for label, mask in tests]
    out = pd.DataFrame(rows)
    valid = np.isfinite(out["p_value"].to_numpy(dtype=float))
    if int(np.sum(valid)) > 0:
        out.loc[valid, "p_fdr"] = multipletests(out.loc[valid, "p_value"], method="fdr_bh")[1]
    else:
        out["p_fdr"] = np.nan
    return out


def build_pairwise_region_contrasts(plot_df: pd.DataFrame) -> pd.DataFrame:
    subj_region = (
        plot_df.groupby(["participant", "region_pair"], observed=True)["mic_z"]
        .mean()
        .reset_index()
        .pivot(index="participant", columns="region_pair", values="mic_z")
    )
    rows = []
    for a, b in combinations(REGION_PAIR_ORDER, 2):
        if a not in subj_region.columns or b not in subj_region.columns:
            continue
        pair_df = subj_region[[a, b]].dropna()
        n_subjects = int(len(pair_df))
        if n_subjects < 2:
            rows.append(
                {
                    "region_a": a,
                    "region_b": b,
                    "mean_diff_a_minus_b": np.nan,
                    "t_stat": np.nan,
                    "p_value": np.nan,
                    "n_subjects": n_subjects,
                }
            )
            continue
        diff = pair_df[a] - pair_df[b]
        t_stat, p_value = ttest_rel(pair_df[a], pair_df[b], nan_policy="omit")
        rows.append(
            {
                "region_a": a,
                "region_b": b,
                "mean_diff_a_minus_b": float(np.nanmean(diff)),
                "t_stat": float(t_stat),
                "p_value": float(p_value),
                "n_subjects": n_subjects,
            }
        )
    out = pd.DataFrame(rows)
    if len(out):
        valid = np.isfinite(out["p_value"].to_numpy(dtype=float))
        out["p_fdr"] = np.nan
        if int(np.sum(valid)) > 0:
            out.loc[valid, "p_fdr"] = multipletests(out.loc[valid, "p_value"], method="fdr_bh")[1]
    return out.sort_values(["p_fdr", "p_value", "region_a", "region_b"], na_position="last").reset_index(drop=True)


In [ ]:
if len(all_mic_df) == 0:
    print('No MIC data available for plotting/modeling.')
else:
    plot_df = all_mic_df.copy()
    plot_df['rpe_cat'] = normalize_rpe_cat(plot_df['rpe_cat'])
    rpe_keep = ['neg', 'neutral', 'pos']
    plot_df = plot_df[plot_df['rpe_cat'].isin(rpe_keep)].copy()
    plot_df['hit_miss'] = plot_df['hit_miss'].astype(str)
    plot_df = plot_df[plot_df['hit_miss'].isin(['hit', 'miss'])].copy()
    plot_df['region_pair'] = plot_df['region_pair'].astype(str)

    coverage_df = (
        plot_df.groupby(['participant', 'region_pair'], observed=True)
        .agg(
            n_trials=('trial_index', 'nunique'),
            n_rows=('trial_index', 'size'),
            label_rate_hit_miss=('hit_miss', lambda s: float(s.notna().mean())),
            label_rate_rpe_cat=('rpe_cat', lambda s: float(s.notna().mean())),
        )
        .reset_index()
    )
    display(coverage_df.head(15))

    subject_summary_df = (
        plot_df.groupby(['participant', 'region_pair'], observed=True)
        .agg(mic_z_mean=('mic_z', 'mean'), n_trials=('trial_index', 'nunique'))
        .reset_index()
    )
    subject_summary_df.to_csv(analysis_paths['summary_csv'], index=False)
    print('Saved:', analysis_paths['summary_csv'])

    fig, ax = plt.subplots(1, 1, figsize=(10, 5))
    sns.pointplot(
        data=plot_df,
        x='region_pair',
        y='mic_z',
        order=REGION_PAIR_ORDER,
        errorbar='se',
        dodge=False,
        ax=ax,
    )
    ax.set_title(f'ACC cue MIC ({CUE_WINDOW[0]}-{CUE_WINDOW[1]} s) by region pair')
    ax.set_xlabel('Region pair')
    ax.set_ylabel('Shuffle-z MIC')
    ax.tick_params(axis='x', rotation=30)
    fig.tight_layout()
    fig.savefig(analysis_paths['region_pair_pdf'], dpi=300, bbox_inches='tight', facecolor='white')
    fig.savefig(analysis_paths['region_pair_png'], dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
    print('Saved:', analysis_paths['region_pair_pdf'])

    fig = sns.catplot(
        data=plot_df,
        x='hit_miss',
        y='mic_z',
        hue='rpe_cat',
        col='region_pair',
        kind='point',
        order=['miss', 'hit'],
        hue_order=['neg', 'neutral', 'pos'],
        col_order=REGION_PAIR_ORDER,
        errorbar='se',
        sharey=True,
        height=3.6,
        aspect=0.95,
    )
    fig.set_titles('{col_name}')
    fig.set_axis_labels('Memory outcome', 'Shuffle-z MIC')
    fig.figure.tight_layout()
    fig.figure.savefig(analysis_paths['hit_rpe_pdf'], dpi=300, bbox_inches='tight', facecolor='white')
    fig.figure.savefig(analysis_paths['hit_rpe_png'], dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
    print('Saved:', analysis_paths['hit_rpe_pdf'])



In [ ]:
if len(all_mic_df) == 0:
    print('Skipping mixed model: no data.')
elif 'plot_df' not in globals():
    print('Run the region-pair plotting cell first to build plot_df.')
else:
    model_df = plot_df.dropna(subset=['mic_z', 'region_pair', 'hit_miss', 'rpe_cat', 'participant']).copy()
    counts_df = (
        model_df.groupby('region_pair', observed=True)
        .agg(n_subjects=('participant', 'nunique'), n_trials=('trial_index', 'nunique'), n_rows=('trial_index', 'size'))
        .reset_index()
    )
    display(counts_df)

    if len(model_df) < MIN_TRIALS_PER_MODEL:
        raise ValueError(f'Insufficient rows for mixed model: {len(model_df)}')

    mixedlm_result, fit_method = fit_primary_mixedlm(model_df)
    print('MixedLM fit method:', fit_method)
    print(mixedlm_result.summary())

    with open(analysis_paths['mixedlm_summary_txt'], 'w') as f:
        f.write(mixedlm_result.summary().as_text())

    fe_table = pd.DataFrame(
        {
            'term': mixedlm_result.fe_params.index,
            'beta': mixedlm_result.fe_params.values,
            'se': mixedlm_result.bse_fe.values,
            'z_value': mixedlm_result.tvalues.loc[mixedlm_result.fe_params.index].values,
            'p_value': mixedlm_result.pvalues.loc[mixedlm_result.fe_params.index].values,
        }
    )
    fe_table.to_csv(analysis_paths['mixedlm_fixed_effects_csv'], index=False)
    display(fe_table)

    omnibus_df = build_omnibus_table(mixedlm_result)
    omnibus_df['n_rows'] = int(len(model_df))
    omnibus_df['n_subjects'] = int(model_df['participant'].nunique())
    omnibus_df.to_csv(analysis_paths['mixedlm_omnibus_csv'], index=False)
    display(omnibus_df)

    pairwise_df = build_pairwise_region_contrasts(plot_df)
    pairwise_df.to_csv(analysis_paths['pairwise_region_csv'], index=False)
    display(pairwise_df.head(20))

    condition_summary_df = (
        plot_df.groupby(['region_pair', 'hit_miss', 'rpe_cat'], observed=True)
        .agg(
            mean_mic_z=('mic_z', 'mean'),
            sem_mic_z=('mic_z', lambda x: float(np.nanstd(x, ddof=1) / np.sqrt(len(x))) if len(x) > 1 else np.nan),
            n_rows=('mic_z', 'size'),
            n_subjects=('participant', 'nunique'),
        )
        .reset_index()
    )
    condition_summary_df.to_csv(analysis_paths['condition_summary_csv'], index=False)
    display(condition_summary_df.head(20))
    print('Saved mixed-model outputs under', analysis_dir)



In [ ]:
print('Cue window:', CUE_WINDOW)
print('Analysis dir:', analysis_dir)
if len(all_mic_df):
    print('Trial rows:', len(all_mic_df))
    print('Subjects:', all_mic_df['participant'].nunique())

